## Step 3  
Input: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/  
Output: s3://thesis--ec331-s3/melted-price-bids/  

In [1]:
# # Test script
# import awswrangler as wr
# import pandas as pd
# import os
# import boto3
# from datetime import datetime

# def test_s3_connection(bucket_name="thesis--ec331-s3"):
#     """Test S3 connection and list objects in bucket"""
#     try:
#         # Create S3 client
#         s3 = boto3.client('s3')
        
#         # List some objects to test connection
#         response = s3.list_objects_v2(
#             Bucket=bucket_name,
#             MaxKeys=5  # Just list a few items to confirm access
#         )
        
#         if 'Contents' in response:
#             print(f"✅ Successfully connected to S3 bucket: {bucket_name}")
#             print(f"Found {len(response['Contents'])} objects. Here are up to 5:")
#             for obj in response['Contents']:
#                 print(f"  - {obj['Key']} ({obj['Size']} bytes)")
#             return True
#         else:
#             print(f"⚠️ Bucket appears empty or you may not have list permissions: {bucket_name}")
#             return False
            
#     except Exception as e:
#         print(f"❌ Error connecting to S3: {e}")
#         return False

# def test_read_single_parquet(file_path):
#     """Test reading a single parquet file from S3"""
#     try:
#         # Read the file
#         start_time = datetime.now()
#         df = wr.s3.read_parquet(path=file_path)
#         end_time = datetime.now()
        
#         # Print statistics
#         print(f"✅ Successfully read Parquet file: {file_path}")
#         print(f"   Time taken: {(end_time - start_time).total_seconds():.2f} seconds")
#         print(f"   DataFrame shape: {df.shape}")
#         print(f"   Columns: {df.columns.tolist()}")
#         print("\nSample data (first 5 rows):")
#         print(df.head())
        
#         return df
#     except Exception as e:
#         print(f"❌ Error reading Parquet file {file_path}: {e}")
#         return None

# def test_list_parquet_files(folder_path):
#     """List all Parquet files in a folder"""
#     try:
#         # List all Parquet files in the folder
#         files = wr.s3.list_objects(path=folder_path, suffix='.parquet')
        
#         print(f"✅ Found {len(files)} Parquet files in: {folder_path}")
#         # Print the first 5 files
#         for i, file in enumerate(files[:5]):
#             print(f"  {i+1}. {file}")
        
#         if len(files) > 5:
#             print(f"  ... and {len(files)-5} more files")
            
#         return files
#     except Exception as e:
#         print(f"❌ Error listing Parquet files in {folder_path}: {e}")
#         return []

# # Main execution
# if __name__ == "__main__":
#     # Test S3 connection
#     print("\n=== Testing S3 Connection ===")
#     test_s3_connection()
    
#     # Test reading from price bids folder
#     print("\n=== Testing Parquet File Listing ===")
#     price_bids_folder = "s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/"
#     files = test_list_parquet_files(price_bids_folder)
    
#     # If files found, test reading one
#     if files:
#         print("\n=== Testing Single Parquet File Reading ===")
#         test_file = files[0]  # Use the first file found
#         test_read_single_parquet(test_file)
        
#         print("\n=== Ready for full processing ===")
#         print("After confirming these tests work, you can add your processing code here.")
#         print("Example:")
#         print("def process_all_files(input_folder, output_folder):")
#         print("    # Your processing logic here")
#         print("    pass")
#         print("")
#         print("# Then call it like:")
#         print("# process_all_files(")
#         print("#     input_folder='s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/',")
#         print("#     output_folder='s3://thesis--ec331-s3/melted-price-bids/'")
#         print("# )")
#     else:
#         print("Cannot proceed with testing as no Parquet files were found.")


=== Testing S3 Connection ===
✅ Successfully connected to S3 bucket: thesis--ec331-s3
Found 5 objects. Here are up to 5:
  - AEMO-Participants - Sheet1.csv (96385 bytes)
  - BIDPEROFFER-CSV/ (0 bytes)
  - BIDPEROFFER-CSV/PUBLIC_DVD_BIDDAYOFFER_202310010000.CSV (1058667104 bytes)
  - BIDPEROFFER-CSV/PUBLIC_DVD_BIDPEROFFER1_202310010000.CSV (57051673387 bytes)
  - BIDPEROFFER-CSV/PUBLIC_DVD_BIDPEROFFER1_202311010000.CSV (67510151107 bytes)

=== Testing Parquet File Listing ===
✅ Found 3 Parquet files in: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/
  1. s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_FILTERED_202310010000.parquet
  2. s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_FILTERED_202311010000.parquet
  3. s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_FILTERED_202312010000.parquet

=== Testing Single Parquet File Reading ===
✅ Successfully read Parquet file: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_FILTERED_202310010000.parq

In [1]:
import pandas as pd
import awswrangler as wr
import time
from datetime import datetime
import gc
import os
import psutil
import boto3

def get_memory_usage():
    """Return the current memory usage of the process in GB"""
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024 / 1024 / 1024
    return memory_gb

print(f"Initial memory usage: {get_memory_usage():.2f} GB")

# Define which columns to melt for price bids
price_band_cols = [f"PRICEBAND{i}" for i in range(1, 11)]

def melt_and_write_chunks(df, chunk_size=50000, timestamp=None, output_folder=""):
    """
    Melts the PRICEBAND columns in chunks and writes each chunk directly to S3.
    Writes files to the given output_folder.
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        
    print(f"Starting streaming melt of dataframe with shape: {df.shape}")
    print(f"Current memory usage: {get_memory_usage():.2f} GB")
    start_time = time.time()
    
    # Calculate number of chunks
    num_chunks = (len(df) + chunk_size - 1) // chunk_size
    print(f"Processing in {num_chunks} chunks of size {chunk_size}")
    
    # Create output path base (for melted files)
    output_base = f"{output_folder}raise1sec_bids_melted_{timestamp}"
    
    # Track total rows processed and file paths
    total_rows_processed = 0
    all_chunk_files = []
    
    # Process dataframe in chunks
    for i in range(num_chunks):
        chunk_start = i * chunk_size
        chunk_end = min((i + 1) * chunk_size, len(df))
        
        print(f"Processing chunk {i+1}/{num_chunks} (rows {chunk_start} to {chunk_end-1})")
        print(f"Memory before chunk processing: {get_memory_usage():.2f} GB")
        
        # Extract and copy chunk
        chunk = df.iloc[chunk_start:chunk_end].copy()
        
        # Identify columns not being melted
        id_vars_cols = [col for col in chunk.columns if col not in price_band_cols]
        # Keep only necessary columns
        chunk = chunk[id_vars_cols + [col for col in price_band_cols if col in chunk.columns]]
        
        # Melt this chunk
        chunk_start_time = time.time()
        chunk_melted = pd.melt(
            chunk,
            id_vars=id_vars_cols,
            value_vars=[col for col in price_band_cols if col in chunk.columns],
            var_name="BIDBAND",
            value_name="BIDPRICE"
        )
        
        # Clean up original chunk to free memory
        del chunk
        gc.collect()
        
        # Extract the band number
        chunk_melted["BIDBAND"] = chunk_melted["BIDBAND"].str.extract(r"(\d+)").astype(int)
        
        # Apply any necessary type transformations
        if "BIDTYPE" in chunk_melted.columns:
            chunk_melted["BIDTYPE"] = chunk_melted["BIDTYPE"].astype(str)
        if "DUID" in chunk_melted.columns:
            chunk_melted["DUID"] = chunk_melted["DUID"].astype(str)
        if "SETTLEMENTDATE" in chunk_melted.columns:
            chunk_melted["SETTLEMENTDATE"] = pd.to_datetime(chunk_melted["SETTLEMENTDATE"])
        
        # Filter out null BIDPRICE values
        chunk_melted = chunk_melted.dropna(subset=["BIDPRICE"])
        
        # Update row count and write this chunk to S3
        chunk_rows = len(chunk_melted)
        total_rows_processed += chunk_rows
        chunk_output = f"{output_base}_part{i+1:04d}.parquet"
        write_start = time.time()
        
        try:
            wr.s3.to_parquet(
                df=chunk_melted,
                path=chunk_output,
                index=False,
                compression="snappy"
            )
            all_chunk_files.append(chunk_output)
            write_time = time.time() - write_start
            print(f"  ✓ Chunk {i+1} written to S3 in {write_time:.2f} seconds ({chunk_rows} rows)")
        except Exception as e:
            print(f"  ✗ Error writing chunk {i+1} to S3: {str(e)}")
        
        # Clean up melted chunk and report memory usage
        del chunk_melted
        gc.collect()
        print(f"  Memory after chunk processing: {get_memory_usage():.2f} GB")
        print(f"  Chunk {i+1} processed in {time.time() - chunk_start_time:.2f} seconds")
    
    total_time = time.time() - start_time
    print(f"All chunks processed and written in {total_time:.2f} seconds")
    print(f"Total rows processed: {total_rows_processed}")
    print(f"Final memory usage: {get_memory_usage():.2f} GB")
    
    return output_base, all_chunk_files, total_rows_processed

def process_s3_files(input_folder="s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/", 
                     batch_size=5,
                     create_single_manifest=True):
    """
    Process Parquet files directly from S3. This function will:
      - Use a fixed base output folder for all outputs:
          s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids_melted/
      - Create a subfolder 'files/' within the base folder for melted files.
      - Create a subfolder 'manifests/' within the base folder for manifest files.
      - Process files in batches, melt the PRICEBAND columns, and write output to the melted files folder.
      - Create manifest files in the manifest folder.
    """
    # Define the base output folder
    base_output = "s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids_melted/"
    # Subfolder for melted files
    output_folder = base_output + "files/"
    # Subfolder for manifest files
    manifest_folder = base_output + "manifests/"
    
    print(f"Base output folder: {base_output}")
    print(f"Output folder for melted files: {output_folder}")
    print(f"Manifest folder for manifest files: {manifest_folder}")
    
    # Create a timestamp for naming consistency
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    
    # Locate all the price bids files in S3 using boto3 for better pagination
    print("Identifying price bids files in S3...")
    try:
        bucket = input_folder.split("/")[2]
        prefix = "/".join(input_folder.split("/")[3:])
        if prefix and not prefix.endswith('/'):
            prefix += '/'
            
        s3_client = boto3.client('s3')
        price_files = []
        
        paginator = s3_client.get_paginator('list_objects_v2')
        page_iterator = paginator.paginate(Bucket=bucket, Prefix=prefix)
        
        for page in page_iterator:
            if 'Contents' in page:
                for obj in page['Contents']:
                    if obj['Key'].endswith('.parquet'):
                        file_path = f"s3://{bucket}/{obj['Key']}"
                        price_files.append(file_path)
        
        print(f"Found {len(price_files)} price bids files")
        if not price_files:
            raise ValueError("No price bids files found in S3")
    except Exception as e:
        print(f"Error listing price bids files: {str(e)}")
        raise
    
    # Process the files in batches to avoid memory issues
    num_batches = (len(price_files) + batch_size - 1) // batch_size
    print(f"Will process files in {num_batches} batches of up to {batch_size} files each")
    
    melted_manifests = []  # To store batch manifest file paths
    all_melted_files = []  # To store all melted file paths
    global_total_rows = 0  # Total rows processed across all batches
    
    for batch_num in range(num_batches):
        batch_start = batch_num * batch_size
        batch_end = min((batch_num + 1) * batch_size, len(price_files))
        batch_files = price_files[batch_start:batch_end]
        
        print(f"\nProcessing batch {batch_num + 1}/{num_batches} with {len(batch_files)} files")
        print(f"Files in this batch: {[os.path.basename(f) for f in batch_files]}")
        print(f"Memory before batch processing: {get_memory_usage():.2f} GB")
        
        try:
            print(f"Reading batch of {len(batch_files)} files...")
            batch_start_time = time.time()
            batch_df = wr.s3.read_parquet(
                path=batch_files,
                dataset=False
            )
            print(f"Read batch in {time.time() - batch_start_time:.2f} seconds")
            print(f"Batch data shape: {batch_df.shape}")
            print(f"Memory after reading batch: {get_memory_usage():.2f} GB")
            
            batch_timestamp = f"{timestamp}_batch{batch_num+1:03d}"
            output_base, batch_melted_files, batch_rows = melt_and_write_chunks(
                batch_df, 
                chunk_size=50000, 
                timestamp=batch_timestamp,
                output_folder=output_folder
            )
            
            all_melted_files.extend(batch_melted_files)
            global_total_rows += batch_rows
            
            del batch_df
            gc.collect()
            
            # Create a manifest file for this batch in the manifest folder
            try:
                manifest = pd.DataFrame({"file_path": batch_melted_files})
                manifest_path = f"{manifest_folder}manifest_{batch_timestamp}.csv"
                wr.s3.to_csv(manifest, manifest_path, index=False)
                print(f"Created manifest file for batch {batch_num + 1}: {manifest_path}")
                melted_manifests.append(manifest_path)
            except Exception as e:
                print(f"Error creating manifest file for batch {batch_num + 1}: {str(e)}")
        except Exception as e:
            print(f"Error processing batch {batch_num + 1}: {str(e)}")
        
        print(f"Completed batch {batch_num + 1}/{num_batches}")
        print(f"Memory after batch processing: {get_memory_usage():.2f} GB")
    
    print("\nAll batches processed!")
    print(f"Total melted files created: {len(all_melted_files)}")
    
    # Create a master manifest of all batch manifests in the manifest folder
    master_manifest_path = None
    single_manifest_path = None
    
    try:
        if melted_manifests:
            master_manifest = pd.DataFrame({"manifest_path": melted_manifests})
            master_manifest_path = f"{manifest_folder}master_manifest_{timestamp}.csv"
            wr.s3.to_csv(master_manifest, master_manifest_path, index=False)
            print(f"Created master manifest at: {master_manifest_path}")
        
        # Create a single manifest file with all processed files
        if create_single_manifest and all_melted_files:
            file_info = []
            for file_path in all_melted_files:
                file_name = os.path.basename(file_path)
                file_info.append({
                    "file_path": file_path,
                    "file_name": file_name,
                    "batch_number": file_name.split("_batch")[1][:3] if "_batch" in file_name else "N/A",
                    "part_number": file_name.split("_part")[1].split(".")[0] if "_part" in file_name else "N/A"
                })
            single_manifest = pd.DataFrame(file_info)
            single_manifest_path = f"{manifest_folder}all_files_manifest_{timestamp}.csv"
            wr.s3.to_csv(single_manifest, single_manifest_path, index=False)
            print(f"Created single manifest with all {len(all_melted_files)} files at: {single_manifest_path}")
    
    except Exception as e:
        print(f"Error creating manifests: {str(e)}")
    
    return master_manifest_path, single_manifest_path, all_melted_files, global_total_rows

# Main execution - Run in your Jupyter cell or script
if __name__ == "__main__":
    input_folder = "s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/"
    master_manifest_path, single_manifest_path, all_melted_files, total_rows = process_s3_files(
        input_folder=input_folder, 
        batch_size=5,
        create_single_manifest=True
    )
    print(f"\nProcessing complete! Total rows processed: {total_rows}")
    print(f"Single manifest file with all outputs: {single_manifest_path}")

Initial memory usage: 0.16 GB
Base output folder: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids_melted/
Output folder for melted files: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids_melted/files/
Manifest folder for manifest files: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids_melted/manifests/
Identifying price bids files in S3...
Found 3 price bids files
Will process files in 1 batches of up to 5 files each

Processing batch 1/1 with 3 files
Files in this batch: ['RAISE1SEC_FILTERED_202310010000.parquet', 'RAISE1SEC_FILTERED_202311010000.parquet', 'RAISE1SEC_FILTERED_202312010000.parquet']
Memory before batch processing: 0.18 GB
Reading batch of 3 files...
Read batch in 1.45 seconds
Batch data shape: (412674, 35)
Memory after reading batch: 0.87 GB
Starting streaming melt of dataframe with shape: (412674, 35)
Current memory usage: 0.87 GB
Processing in 9 chunks of size 50000
Processing chunk 1/9 (rows 0 to 49999)
Memory before chunk processing: 0.87 GB
  ✓ Chunk 1 written to S3

In [3]:
import pandas as pd
import awswrangler as wr
import time
from datetime import datetime
import gc
import os
import psutil
import boto3

def get_memory_usage():
    """Return the current memory usage of the process in GB"""
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024 / 1024 / 1024
    return memory_gb

print(f"Initial memory usage: {get_memory_usage():.2f} GB")

# Define which columns to melt for price bids
price_band_cols = [f"PRICEBAND{i}" for i in range(1, 11)]

def melt_and_write_chunks(df, chunk_size=50000, timestamp=None, output_folder=""):
    """
    Melts the PRICEBAND columns in chunks and writes each chunk directly to S3.
    Writes files to the given output_folder.
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        
    print(f"Starting streaming melt of dataframe with shape: {df.shape}")
    print(f"Current memory usage: {get_memory_usage():.2f} GB")
    start_time = time.time()
    
    # Calculate number of chunks
    num_chunks = (len(df) + chunk_size - 1) // chunk_size
    print(f"Processing in {num_chunks} chunks of size {chunk_size}")
    
    # Create output path base (for melted files)
    output_base = f"{output_folder}raise1sec_bids_melted_{timestamp}"
    
    # Track total rows processed and file paths
    total_rows_processed = 0
    all_chunk_files = []
    
    # Process dataframe in chunks
    for i in range(num_chunks):
        chunk_start = i * chunk_size
        chunk_end = min((i + 1) * chunk_size, len(df))
        
        print(f"Processing chunk {i+1}/{num_chunks} (rows {chunk_start} to {chunk_end-1})")
        print(f"Memory before chunk processing: {get_memory_usage():.2f} GB")
        
        # Extract and copy chunk
        chunk = df.iloc[chunk_start:chunk_end].copy()
        
        # Identify columns not being melted
        id_vars_cols = [col for col in chunk.columns if col not in price_band_cols]
        # Keep only necessary columns
        chunk = chunk[id_vars_cols + [col for col in price_band_cols if col in chunk.columns]]
        
        # Melt this chunk
        chunk_start_time = time.time()
        chunk_melted = pd.melt(
            chunk,
            id_vars=id_vars_cols,
            value_vars=[col for col in price_band_cols if col in chunk.columns],
            var_name="BIDBAND",
            value_name="BIDPRICE"
        )
        
        # Clean up original chunk to free memory
        del chunk
        gc.collect()
        
        # Extract the band number
        chunk_melted["BIDBAND"] = chunk_melted["BIDBAND"].str.extract(r"(\d+)").astype(int)
        
        # Apply any necessary type transformations
        if "BIDTYPE" in chunk_melted.columns:
            chunk_melted["BIDTYPE"] = chunk_melted["BIDTYPE"].astype(str)
        if "DUID" in chunk_melted.columns:
            chunk_melted["DUID"] = chunk_melted["DUID"].astype(str)
        if "SETTLEMENTDATE" in chunk_melted.columns:
            chunk_melted["SETTLEMENTDATE"] = pd.to_datetime(chunk_melted["SETTLEMENTDATE"])
        
        # Filter out null BIDPRICE values
        chunk_melted = chunk_melted.dropna(subset=["BIDPRICE"])
        
        # Update row count and write this chunk to S3
        chunk_rows = len(chunk_melted)
        total_rows_processed += chunk_rows
        chunk_output = f"{output_base}_part{i+1:04d}.parquet"
        write_start = time.time()
        
        try:
            wr.s3.to_parquet(
                df=chunk_melted,
                path=chunk_output,
                index=False,
                compression="snappy"
            )
            all_chunk_files.append(chunk_output)
            write_time = time.time() - write_start
            print(f"  ✓ Chunk {i+1} written to S3 in {write_time:.2f} seconds ({chunk_rows} rows)")
        except Exception as e:
            print(f"  ✗ Error writing chunk {i+1} to S3: {str(e)}")
        
        # Clean up melted chunk and report memory usage
        del chunk_melted
        gc.collect()
        print(f"  Memory after chunk processing: {get_memory_usage():.2f} GB")
        print(f"  Chunk {i+1} processed in {time.time() - chunk_start_time:.2f} seconds")
    
    total_time = time.time() - start_time
    print(f"All chunks processed and written in {total_time:.2f} seconds")
    print(f"Total rows processed: {total_rows_processed}")
    print(f"Final memory usage: {get_memory_usage():.2f} GB")
    
    return output_base, all_chunk_files, total_rows_processed

def process_single_file(input_file):
    """
    Process a single Parquet file from S3. This function will:
      - Extract the file name and use it to create a folder in the output path
      - Melt the PRICEBAND columns and write the output to the created folder
      - Create a manifest file for the processed file
    """
    print(f"Processing single file: {input_file}")
    
    # Extract the file name without extension to use as folder name
    file_name = os.path.basename(input_file)
    file_name_no_ext = os.path.splitext(file_name)[0]
    
    # Define the base output folder
    base_output = "s3://thesis--ec331-s3/melted-price-bids/"
    
    # Create a folder with the same name as the file
    output_folder = f"{base_output}{file_name_no_ext}/"
    
    # Subfolder for manifest files
    manifest_folder = f"{output_folder}manifest/"
    
    print(f"Base output folder: {base_output}")
    print(f"Output folder for melted files: {output_folder}")
    print(f"Manifest folder for manifest files: {manifest_folder}")
    
    # Create a timestamp for naming consistency
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    
    try:
        print(f"Reading file: {input_file}")
        start_time = time.time()
        df = wr.s3.read_parquet(path=input_file)
        print(f"Read file in {time.time() - start_time:.2f} seconds")
        print(f"Data shape: {df.shape}")
        print(f"Memory after reading file: {get_memory_usage():.2f} GB")
        
        # Process the file
        output_base, melted_files, total_rows = melt_and_write_chunks(
            df, 
            chunk_size=50000, 
            timestamp=timestamp,
            output_folder=output_folder
        )
        
        del df
        gc.collect()
        
        # Create a manifest file in the manifest folder
        try:
            manifest = pd.DataFrame({
                "file_path": melted_files,
                "file_name": [os.path.basename(f) for f in melted_files],
                "part_number": [os.path.basename(f).split("_part")[1].split(".")[0] 
                               if "_part" in os.path.basename(f) else "N/A" 
                               for f in melted_files]
            })
            manifest_path = f"{manifest_folder}manifest_{timestamp}.csv"
            
            # Ensure the manifest directory exists
            s3_client = boto3.client('s3')
            s3_client.put_object(
                Bucket=manifest_path.split('/')[2],
                Key='/'.join(manifest_path.split('/')[3:-1]) + '/',
                Body=''
            )
            
            wr.s3.to_csv(manifest, manifest_path, index=False)
            print(f"Created manifest file: {manifest_path}")
        except Exception as e:
            print(f"Error creating manifest file: {str(e)}")
            
        return melted_files, total_rows
        
    except Exception as e:
        print(f"Error processing file: {str(e)}")
        raise

# Main execution
if __name__ == "__main__":
    # The single file to process
    input_file = "s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_FILTERED_202310010000.parquet"
    
    melted_files, total_rows = process_single_file(input_file)
    
    print(f"\nProcessing complete! Total rows processed: {total_rows}")
    print(f"Total melted files created: {len(melted_files)}")

Initial memory usage: 0.30 GB
Processing single file: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_FILTERED_202310010000.parquet
Base output folder: s3://thesis--ec331-s3/melted-price-bids/
Output folder for melted files: s3://thesis--ec331-s3/melted-price-bids/RAISE1SEC_FILTERED_202310010000/
Manifest folder for manifest files: s3://thesis--ec331-s3/melted-price-bids/RAISE1SEC_FILTERED_202310010000/manifest/
Reading file: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_FILTERED_202310010000.parquet
Read file in 0.53 seconds
Data shape: (103162, 35)
Memory after reading file: 0.44 GB
Starting streaming melt of dataframe with shape: (103162, 35)
Current memory usage: 0.44 GB
Processing in 3 chunks of size 50000
Processing chunk 1/3 (rows 0 to 49999)
Memory before chunk processing: 0.44 GB
  ✓ Chunk 1 written to S3 in 2.14 seconds (500000 rows)
  Memory after chunk processing: 0.59 GB
  Chunk 1 processed in 4.29 seconds
Processing chunk 2/3 (rows 50000 to 99999)
Me

In [ ]:
price_bids_df.columns

In [ ]:
# To find the first (earliest) datetime
first_datetime = price_bids_df['SETTLEMENTDATE'].min()

# To find the last (latest) datetime
last_datetime = price_bids_df['SETTLEMENTDATE'].max()

# Print the results
print(f"First datetime: {first_datetime}")
print(f"Last datetime: {last_datetime}")

In [4]:
import pandas as pd
import awswrangler as wr

# Path to the test sample
test_sample_path = "s3://thesis--ec331-s3/melted-price-bids/test_melted_sample.parquet"

# Read the file
try:
    df = wr.s3.read_parquet(path=test_sample_path)
    
    # Display basic information
    print(f"Successfully read file: {test_sample_path}")
    print(f"Shape: {df.shape}")
    print("\nColumns:")
    print(df.columns.tolist())
    
    # Display a sample of rows
    print("\nSample data:")
    display(df.head(10))
    
    # Check value distributions if sample is large enough
    if len(df) > 10:
        print("\nBIDBBAND distribution:")
        display(df['BIDBAND'].value_counts().sort_index())
        
        # Get basic statistics on BIDPRICE
        print("\nBIDPRICE statistics:")
        display(df['BIDPRICE'].describe())
except Exception as e:
    print(f"Error reading Parquet file: {e}")

Successfully read file: s3://thesis--ec331-s3/melted-price-bids/test_melted_sample.parquet
Shape: (10000, 27)

Columns:
['I', 'BIDS', 'BIDDAYOFFER', '1', 'DUID', 'BIDTYPE', 'SETTLEMENTDATE', 'OFFERDATE', 'VERSIONNO', 'PARTICIPANTID', 'DAILYENERGYCONSTRAINT', 'REBIDEXPLANATION', 'MINIMUMLOAD', 'T1', 'T2', 'T3', 'T4', 'NORMALSTATUS', 'LASTCHANGED', 'ENTRYTYPE', 'REBID_EVENT_TIME', 'REBID_AWARE_TIME', 'REBID_DECISION_TIME', 'REBID_CATEGORY', 'REFERENCE_ID', 'BIDBAND', 'BIDPRICE']

Sample data:


,I,BIDS,BIDDAYOFFER,1,DUID,BIDTYPE,SETTLEMENTDATE,OFFERDATE,VERSIONNO,PARTICIPANTID,...,NORMALSTATUS,LASTCHANGED,ENTRYTYPE,REBID_EVENT_TIME,REBID_AWARE_TIME,REBID_DECISION_TIME,REBID_CATEGORY,REFERENCE_ID,BIDBAND,BIDPRICE
0,D,BIDS,BIDDAYOFFER,1.0,ASNAES1,RAISE1SEC,2023/10/29 00:00:00,2023/10/27 14:31:21,1.0,AMALMASP,...,NaN,2023/10/27 14:31:21,DAILY,<NA>,<NA>,<NA>,<NA>,BE737C9E9E234F998B18FE4B2F9AC2E8,1,0.0
1,D,BIDS,BIDDAYOFFER,1.0,ASNAES1,RAISE1SEC,2023/10/31 00:00:00,2023/10/30 11:02:53,1.0,AMALMASP,...,NaN,2023/10/30 11:02:53,DAILY,<NA>,<NA>,<NA>,<NA>,a529df1f-c7ac-45c5-8995-4fae5b45ce46,1,0.0
2,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/08 00:00:00,2023/10/05 15:35:07,1.0,ENOCMASP,...,NaN,2023/10/05 15:35:07,DAILY,<NA>,<NA>,<NA>,<NA>,2370F99E2EF34763BC4001A791AD49B9,1,0.0
3,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/09 00:00:00,2023/10/06 14:19:35,1.0,ENOCMASP,...,NaN,2023/10/06 14:19:35,DAILY,15:15:00,<NA>,<NA>,<NA>,8f8661ba-0579-4ccb-9c99-80ba0f1f9a5d,1,0.0
4,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/09 00:00:00,2023/10/09 13:10:27,1.0,ENOCMASP,...,NaN,2023/10/09 13:10:27,REBID,13:08:00,<NA>,<NA>,<NA>,fb4c0f11-359e-47bf-a0ab-634a117b4390,1,0.0
5,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/10 00:00:00,2023/10/09 13:48:09,1.0,ENOCMASP,...,NaN,2023/10/09 13:48:09,REBID,13:45:00,<NA>,<NA>,<NA>,99c77c3c-d57e-4728-a8cc-af35e4dac82a,1,0.0
6,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/11 00:00:00,2023/10/10 08:34:44,1.0,ENOCMASP,...,NaN,2023/10/10 08:34:44,DAILY,08:30:00,<NA>,<NA>,<NA>,130b5977-aab4-4e62-b10d-66bc0d092b0c,1,0.5
7,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/12 00:00:00,2023/10/11 08:15:20,1.0,ENOCMASP,...,NaN,2023/10/11 08:15:20,DAILY,08:15:00,<NA>,<NA>,<NA>,feba9199-5ad5-4d39-bc75-5a23374e106d,1,0.5
8,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/12 00:00:00,2023/10/11 14:08:46,1.0,ENOCMASP,...,NaN,2023/10/11 14:08:46,REBID,14:05:00,<NA>,<NA>,<NA>,8e22789f-0ac2-462a-95c6-de2cbcf8c3dc,1,0.5
9,D,BIDS,BIDDAYOFFER,1.0,ASNENC1,RAISE1SEC,2023/10/12 00:00:00,2023/10/12 09:49:17,1.0,ENOCMASP,...,NaN,2023/10/12 09:49:17,REBID,09:48:00,09:48:00,09:48:00,E,272b4135-b2ab-4f93-882a-07eab01f31da,1,0.5



BIDBBAND distribution:


BIDBAND
1     1000
2     1000
3     1000
4     1000
5     1000
6     1000
7     1000
8     1000
9     1000
10    1000
Name: count, dtype: Int64


BIDPRICE statistics:


count    10000.000000
mean      1603.498327
std       4779.386097
min          0.000000
25%          1.160000
50%          8.000000
75%         49.950000
max      16600.000000
Name: BIDPRICE, dtype: float64

Successfully read manifest file with 9 entries


,file_path,file_name,batch_number,part_number
0,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,1
1,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,2
2,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,3
3,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,4
4,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,5
5,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,6
6,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,7
7,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,8
8,s3://thesis--ec331-s3/melted-price-bids/raise1...,raise1sec_bids_melted_20250315134512_batch001_...,1,9


In [ ]:
s3://thesis--ec331-s3/melted-price-bids/
